# Access Log NASA

# **Buka dan Baca File Log**

Di cell ini, kita membuka file log web yang berisi catatan aktivitas server. Semua baris di dalam file dibaca dan disimpan ke dalam variabel data

In [ ]:
with open("/content/access_log_Jul95", 'r', encoding='latin-1') as f:
    data = f.readlines()

print(data[:5])

['199.72.81.55 - - [01/Jul/1995:00:00:01 -0400] "GET /history/apollo/ HTTP/1.0" 200 6245\n', 'unicomp6.unicomp.net - - [01/Jul/1995:00:00:06 -0400] "GET /shuttle/countdown/ HTTP/1.0" 200 3985\n', '199.120.110.21 - - [01/Jul/1995:00:00:09 -0400] "GET /shuttle/missions/sts-73/mission-sts-73.html HTTP/1.0" 200 4085\n', 'burger.letters.com - - [01/Jul/1995:00:00:11 -0400] "GET /shuttle/countdown/liftoff.html HTTP/1.0" 304 0\n', '199.120.110.21 - - [01/Jul/1995:00:00:11 -0400] "GET /shuttle/missions/sts-73/sts-73-patch-small.gif HTTP/1.0" 200 4179\n']


# **Memecah Baris Log Jadi Data Terstruktur**

Cell ini bertugas memecah setiap baris log menjadi bagian-bagian penting menggunakan pola regex. Jadi dari satu baris log panjang, kita ambil infonya seperti IP, waktu akses, jenis request (GET/POST), halaman yang diakses, status kode, dan ukuran respon.

In [ ]:
import re

log_pattern = re.compile(r'(\S+) - - \[(.*?)\] "(\S+) (.*?) (\S+)" (\S+) (\S+)')
parsed_data = []

for line in data:
    match = log_pattern.match(line)
    if match:
        hostname, timestamp, request_type, path, protocol, status_code, size = match.groups()
        parsed_data.append([hostname, timestamp, request_type, path, status_code, size])
    else:
        # Optional: Log lines that don't match the pattern
        # print(f"Skipping line: {line.strip()}")
        pass

print(parsed_data[:5])

[['199.72.81.55', '01/Jul/1995:00:00:01 -0400', 'GET', '/history/apollo/', '200', '6245'], ['unicomp6.unicomp.net', '01/Jul/1995:00:00:06 -0400', 'GET', '/shuttle/countdown/', '200', '3985'], ['199.120.110.21', '01/Jul/1995:00:00:09 -0400', 'GET', '/shuttle/missions/sts-73/mission-sts-73.html', '200', '4085'], ['burger.letters.com', '01/Jul/1995:00:00:11 -0400', 'GET', '/shuttle/countdown/liftoff.html', '304', '0'], ['199.120.110.21', '01/Jul/1995:00:00:11 -0400', 'GET', '/shuttle/missions/sts-73/sts-73-patch-small.gif', '200', '4179']]


# **Menampilkan ke Dataframe**

In [ ]:
import pandas as pd

df = pd.DataFrame(parsed_data, columns=['hostname', 'timestamp', 'request_type', 'path', 'status_code', 'size'])
display(df.head())

,hostname,timestamp,request_type,path,status_code,size
0,199.72.81.55,01/Jul/1995:00:00:01 -0400,GET,/history/apollo/,200,6245
1,unicomp6.unicomp.net,01/Jul/1995:00:00:06 -0400,GET,/shuttle/countdown/,200,3985
2,199.120.110.21,01/Jul/1995:00:00:09 -0400,GET,/shuttle/missions/sts-73/mission-sts-73.html,200,4085
3,burger.letters.com,01/Jul/1995:00:00:11 -0400,GET,/shuttle/countdown/liftoff.html,304,0
4,199.120.110.21,01/Jul/1995:00:00:11 -0400,GET,/shuttle/missions/sts-73/sts-73-patch-small.gif,200,4179


## **Jumlah Data**

In [ ]:
len(df.timestamp)

1888722

## **IP yang banyak di Akses**

In [ ]:
# === 2️⃣ Tampilkan jumlah duplikasi berdasarkan IP ===
dupe_count = df["hostname"].value_counts()
print("📊 Jumlah kemunculan setiap IP:")
display(dupe_count.head(10))  # tampilkan 10 IP teratas


📊 Jumlah kemunculan setiap IP:


,count
hostname,
piweba3y.prodigy.com,17572
piweba4y.prodigy.com,11591
piweba1y.prodigy.com,9868
alyssa.prodigy.com,7852
siltb10.orl.mmc.com,7573
piweba2y.prodigy.com,5922
edams.ksc.nasa.gov,5434
163.206.89.4,4906
news.ti.com,4863


## **Ubah Waktu jadi format Datetime**

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%d/%b/%Y:%H:%M:%S %z')
df_sorted = df.sort_values(by=['hostname', 'timestamp'])
display(df_sorted.head())

,hostname,timestamp,request_type,path,status_code,size
726764,***.novo.dk,1995-07-11 08:17:09-04:00,GET,/ksc.html,200,7067
726765,***.novo.dk,1995-07-11 08:17:11-04:00,GET,/images/ksclogo-medium.gif,200,5866
726787,***.novo.dk,1995-07-11 08:17:31-04:00,GET,/images/MOSAIC-logosmall.gif,200,363
726791,***.novo.dk,1995-07-11 08:17:33-04:00,GET,/images/USA-logosmall.gif,200,234
726793,***.novo.dk,1995-07-11 08:17:34-04:00,GET,/images/NASA-logosmall.gif,200,786


## **Hitung Jenis Request**

In [ ]:
request_type_counts = df['request_type'].value_counts()
print("Nilai unik dalam kolom 'request_type' beserta jumlahnya:")
display(request_type_counts)

Nilai unik dalam kolom 'request_type' beserta jumlahnya:


,count
request_type,
GET,1884659
HEAD,3952
POST,111


## **Status Kode Server**

In [ ]:
request_type_counts = df['status_code'].value_counts()
print("Nilai unik dalam kolom 'status_code' beserta jumlahnya:")
display(request_type_counts)

Nilai unik dalam kolom 'status_code' beserta jumlahnya:


,count
status_code,
200,1698643
304,132627
302,46548
404,10774
500,62
403,54
501,14


memfilter log untuk mencari request POST yang menuju file dengan akhiran .html

In [ ]:
post_html_count = df[(df['request_type'] == 'POST') & (df['path'].str.endswith('.html'))].shape[0]
print(f"Jumlah permintaan POST dengan path diakhiri '.html': {post_html_count}")

Jumlah permintaan POST dengan path diakhiri '.html': 9


memfilter log untuk mencari request GET yang menuju file dengan akhiran .html

In [ ]:
post_html_count = df[(df['request_type'] == 'GET') & (df['path'].str.endswith('.html'))].shape[0]
print(f"Jumlah permintaan POST dengan path diakhiri '.html': {post_html_count}")

Jumlah permintaan POST dengan path diakhiri '.html': 416446


In [ ]:
filtered_df = df[(df['request_type'] == 'GET') & (df['path'].str.endswith('.html')) & (df['status_code'] == '200')]
filtered_df

,hostname,timestamp,request_type,path,status_code,size
2,199.120.110.21,1995-07-01 00:00:09-04:00,GET,/shuttle/missions/sts-73/mission-sts-73.html,200,4085
7,205.212.115.106,1995-07-01 00:00:12-04:00,GET,/shuttle/countdown/countdown.html,200,3985
18,ppptky391.asahi-net.or.jp,1995-07-01 00:00:18-04:00,GET,/facts/about_ksc.html,200,3977
22,waters-gw.starway.net.au,1995-07-01 00:00:25-04:00,GET,/shuttle/missions/51-l/mission-51-l.html,200,6723
37,gayle-gaston.tenet.edu,1995-07-01 00:00:50-04:00,GET,/shuttle/missions/sts-71/mission-sts-71.html,200,12040
...,...,...,...,...,...,...
1888697,alyssa.prodigy.com,1995-07-28 13:32:14-04:00,GET,/shuttle/countdown/liftoff.html,200,5220
1888701,maynard.isi.uconn.edu,1995-07-28 13:32:17-04:00,GET,/shuttle/technology/sts-newsref/sts_egress.html,200,86379
1888703,tornado.umd.edu,1995-07-28 13:32:18-04:00,GET,/shuttle/missions/sts-74/mission-sts-74.html,200,3790
1888705,gk-east.usps.gov,1995-07-28 13:32:19-04:00,GET,/facts/faq.html,200,18290


In [ ]:
filtered_df_sorted = filtered_df.sort_values(by=['hostname', 'timestamp'])
display(filtered_df_sorted.head())

,hostname,timestamp,request_type,path,status_code,size
726764,***.novo.dk,1995-07-11 08:17:09-04:00,GET,/ksc.html,200,7067
726817,***.novo.dk,1995-07-11 08:17:48-04:00,GET,/shuttle/missions/missions.html,200,8678
727034,***.novo.dk,1995-07-11 08:21:05-04:00,GET,/shuttle/missions/sts-35/mission-sts-35.html,200,12118
727042,***.novo.dk,1995-07-11 08:21:19-04:00,GET,/shuttle/missions/sts-35/mission-sts-35.html,200,12118
727149,***.novo.dk,1995-07-11 08:23:01-04:00,GET,/shuttle/resources/orbiters/columbia.html,200,6922


In [ ]:
filtered_df_sorted.to_csv('/content/drive/MyDrive/TUGAS7PPW/PengolahanLogFileNasa/filtered_sorted_log_data.csv', index=False)
print("DataFrame telah disimpan ke 'filtered_sorted_log_data.csv'")

DataFrame telah disimpan ke 'filtered_sorted_log_data.csv'
